# TCN-Transformer Log Anomaly Notebook

Notebook-first запуск проекта. В тетрадке остаются выбор конфига, запуск этапов и просмотр метрик; обучение, SPOT, validation safety-check, ensemble, метрики и визуализации лежат в `src/` и `scripts/`.

Правило проекта: код графиков не пишем в notebook. Картинки генерируются через `scripts/07_generate_report_assets.py`, а реализация находится в `src/visualization/plots.py`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config
from src.utils.notebook_workflow import dataset_links_table, load_metric_tables, run_full_pipeline, run_stage

PROJECT_ROOT

## 1. Датасеты

Ссылки и целевые папки. После скачивания распакуй данные в указанную `target_dir`, затем запускай подготовку.

In [ ]:
dataset_links_table()

## 2. Конфиг эксперимента

Выбери один из конфигов. После `prepare_data` используй `data/processed/<dataset>/used_config.yaml`, потому что там уже записан `model.vocab_size`.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'experiment_lo2.yaml'
config = load_config(CONFIG_PATH)
config

## 3. Подготовка данных

Этот этап вызывает `scripts/01_prepare_data.py`, а сам parsing/adapters/vocab берутся из `src/data/`.

In [ ]:
# Запускай после того, как данные лежат в data/raw/<dataset>/
# run_stage('prepare_data', CONFIG_PATH)

USED_CONFIG_PATH = PROJECT_ROOT / config['data']['processed_dir'] / 'used_config.yaml'
USED_CONFIG_PATH

## 4. Полный pipeline из notebook

`run_full_pipeline` вызывает все проектные этапы: baseline TCN-AE, TCN-Transformer-AE, TCN-Transformer ensemble, detection, localization, adaptive threshold и генерацию report assets.

In [ ]:
# После prepare_data лучше переключиться на used_config:
# CONFIG_FOR_RUN = USED_CONFIG_PATH
CONFIG_FOR_RUN = CONFIG_PATH

# Полный запуск. Если prepare_data уже выполнен, поставь include_prepare=False.
# run_full_pipeline(CONFIG_FOR_RUN, include_prepare=True)

## 5. Поштучный запуск этапов

Если не хочется гонять всё сразу, запускай этапы отдельно. Ensemble и SPOT/safety-check применяются в проектном коде: `scripts/08_train_tcn_transformer_ensemble.py`, `src/evaluation/ensemble.py`, `src/evaluation/thresholds.py`, `scripts/04_evaluate_detection.py`.

In [ ]:
# run_stage('train_tcn_ae', CONFIG_FOR_RUN)
# run_stage('train_tcn_transformer_ae', CONFIG_FOR_RUN)
# run_stage('train_tcn_transformer_ensemble', CONFIG_FOR_RUN)
# run_stage('evaluate_detection', CONFIG_FOR_RUN)
# run_stage('evaluate_localization', CONFIG_FOR_RUN)
# run_stage('evaluate_adaptive_threshold', CONFIG_FOR_RUN)
# run_stage('generate_report_assets', CONFIG_FOR_RUN)

## 6. Все метрики

Таблицы читаются из `outputs/metrics/`. Картинки лежат в `outputs/figures/`, а код их построения находится в `src/visualization/plots.py`.

In [ ]:
tables = load_metric_tables(config.get('project', {}).get('output_dir', 'outputs'))
tables.keys()

In [ ]:
tables.get('detection')

In [ ]:
tables.get('localization')

In [ ]:
tables.get('adaptive_threshold')

In [ ]:
tables.get('final_comparison')

## 7. Где лежит результат

- `outputs/models/` — checkpoints одиночных моделей и ensemble members.
- `outputs/predictions/` — scores, thresholds, predictions.
- `outputs/metrics/` — detection/localization/adaptive/final CSV.
- `outputs/figures/` — готовые PNG из `src/visualization/plots.py`.
- `outputs/reports/summary.md` — markdown-сводка эксперимента.